# Data cleaning workflow\n
\n
Notebook for practical cleaning steps: missing values, duplicates, type conversions and simple outlier filtering.

In [ ]:
from pathlib import Path\n
\n
import numpy as np\n
import pandas as pd

In [ ]:
processed_dir = Path('../data/processed')\n
processed_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Demo-data with typical quality issues\n
df = pd.DataFrame({\n
    'id': [1, 2, 2, 3, 4, 5, 6],\n
    'name': ['Anna', 'Bo', 'Bo', 'Carla', None, 'David', 'Eva'],\n
    'age': [25, 31, 31, np.nan, 42, 120, 29],\n
    'salary': ['35000', '42000', '42000', '39000', '47000', '999999', None],\n
    'signup_date': ['2026-01-10', '2026-02-11', '2026-02-11', 'bad-date', '2026-03-01', '2026-03-20', '2026-03-25'],\n
})\n
\n
df

In [ ]:
# 1) Inspect quality\n
display(df.info())\n
display(df.isna().sum())\n
display(df.duplicated().sum())

In [ ]:
# 2) Remove duplicate rows\n
df_clean = df.drop_duplicates().copy()\n
df_clean

In [ ]:
# 3) Convert types safely\n
df_clean['salary'] = pd.to_numeric(df_clean['salary'], errors='coerce')\n
df_clean['signup_date'] = pd.to_datetime(df_clean['signup_date'], errors='coerce')\n
df_clean

In [ ]:
# 4) Handle missing values (simple strategy)\n
df_clean['name'] = df_clean['name'].fillna('Unknown')\n
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())\n
df_clean['salary'] = df_clean['salary'].fillna(df_clean['salary'].median())\n
df_clean

In [ ]:
# 5) Remove obvious outliers with IQR on salary\n
q1 = df_clean['salary'].quantile(0.25)\n
q3 = df_clean['salary'].quantile(0.75)\n
iqr = q3 - q1\n
lower = q1 - 1.5 * iqr\n
upper = q3 + 1.5 * iqr\n
\n
df_filtered = df_clean[(df_clean['salary'] >= lower) & (df_clean['salary'] <= upper)].copy()\n
df_filtered

In [ ]:
# 6) Final quality check\n
display(df_filtered.info())\n
display(df_filtered.isna().sum())\n
display(df_filtered.describe(include='all'))

In [ ]:
# 7) Export cleaned dataset\n
out_path = processed_dir / 'cleaned_data.csv'\n
df_filtered.to_csv(out_path, index=False)\n
out_path